# Local comparison of single-cell models

This lesson compares every injectable neuron class provided by the installed **PyNN/Brian2** backend and the **five native Brian2 cell adapters** already used by CNT's dashboard. One isolated cell is run per model, with a current step and native model parameters.

The dashboard uses the same `compare_cells()` function. This notebook is a new local experiment; it does not replace the original [synaptic-input / SpiNNaker comparison](eg_testing-cell-models-network.ipynb). The four SpiNNaker-only `extra_models` stay in that notebook. The original [STDP lesson](eg_testing-STDP-model-network.ipynb) also requires the SpiNNaker kernel for its dendritic-delay setting.

## Environment and model catalogue

Install CNT's root `requirements.txt` in your course environment, then open this notebook with that environment's Python kernel. No SpiNNaker account or hardware is needed for the models below.

In [ ]:
from pathlib import Path
import sys
import json
import csv
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Works from the repository root or from notebooks/intro.
repository = next(path for path in [Path.cwd(), *Path.cwd().parents]
                  if (path / "dashboard" / "cnt_dashboard" / "cell_comparison.py").exists())
sys.path.insert(0, str(repository / "dashboard"))
from cnt_dashboard.cell_comparison import comparison_models, compare_cells
from cnt_dashboard.notebook_catalogue import HARDWARE_MODELS

models = comparison_models()
display(pd.DataFrame([{"label": label, "backend": backend, "model": name}
                      for label, (backend, name) in models.items()]))
print(f"{len(models)} local entries; distinct parameterizations/backends are kept separate.")
print("SpiNNaker-only examples in the original notebook:", ", ".join(HARDWARE_MODELS))

## Protocol and units

The dimensional models receive the same physical current in **nA**, converted to pA or uA where required by the CNT adapters. Equal current does not imply equal current density, I/C, firing rate or excitability: their native capacitances and other parameters differ. In particular, the Morris–Lecar teaching preset uses much larger absolute capacitance/conductance scales.

CNT's standalone Izhikevich adapter declares its input in **model units**. It receives a separate amplitude; no conversion from those units to amperes is assumed. PyNN's Izhikevich implementation is a separate native parameterization (its current-to-voltage equation uses a 1 pF scaling).

This lesson has no presynaptic spike trains. Alpha/exponential synaptic variants can therefore produce identical voltage responses. To study synaptic kernels, return to the original input-spike notebook and use a compatible backend.

In [ ]:
protocol = dict(duration_ms=400.0, start_ms=50.0, stop_ms=350.0,
                amplitude_nA=1.0, izh_model_input=10.0, pynn_dt_ms=0.1)
selected = list(models)  # Replace with a subset of catalogue labels for a faster run.
result = compare_cells(**protocol, selected=selected)
print("Completed:", len(result["populations"]), "models")
print("PyNN", result["configuration"]["pynn_version"],
      "/ Brian2", result["configuration"]["brian2_version"])

## Voltage signatures

Each model has its own voltage axis and retains its native integration step. The native adapters use their validated teaching steps; PyNN treats `pynn_dt_ms` as a maximum: HH is capped at 0.01 ms, AdEx variants at 0.025 ms, Izhikevich at 0.05 ms, and other models at 0.1 ms. The actual step is exported for each model. Threshold/reset models do not generate a biophysical action-potential waveform. A scaled Izhikevich trace is labeled with its model-voltage convention.

In [ ]:
names = list(result["populations"])
fig = make_subplots(rows=math.ceil(len(names)/3), cols=3, subplot_titles=names,
                    vertical_spacing=0.07)
for index, (name, population) in enumerate(result["populations"].items()):
    row, col = index//3 + 1, index%3 + 1
    fig.add_trace(go.Scattergl(x=population["time_ms"],
                  y=population["signals"]["v"][:, 0], name=name, mode="lines"), row=row, col=col)
    fig.update_yaxes(title_text=population["voltage_unit"], row=row, col=col)
    fig.update_xaxes(title_text="Time [ms]", row=row, col=col)
fig.update_layout(template="plotly_dark", showlegend=False,
                  height=260*math.ceil(len(names)/3), width=1250)
display(fig)

## Recorded events and input scales

Event counts follow each backend's detector. In particular, native PyNN `HH_cond_exp` can report repeated above-threshold events during one action potential; CNT's HH adapter counts crossings. These counts must not be interpreted as interchangeable action-potential rates. Compare voltage signatures and inspect the detector/reset definitions as well as the count.

In [ ]:
summary = pd.DataFrame([
    {"model": name, "recorded_events": len(population["spikes_ms"][0]),
     "input_amplitude": population["input_amplitude"], "input_unit": population["input_unit"],
     "voltage_unit": population["voltage_unit"], "dt_ms": population["dt_ms"],
     "voltage_min": float(population["signals"]["v"].min()),
     "voltage_max": float(population["signals"]["v"].max())}
    for name, population in result["populations"].items()])
display(summary)

## Inspect the native implementation

The dashboard's model inspector shows the equations and references. Here the following cell exposes the exact PyNN/Brian2 equation strings, default parameters, initial values and event rule for one catalogue entry. For a CNT adapter, inspect `dashboard/cnt_dashboard/cells.py` and `model_notes.py`.

In [ ]:
import pyNN.brian2 as sim
import inspect
pynn_example = next(name for backend, name in models.values() if backend == "pynn")
model_class = getattr(sim, pynn_example)
print(pynn_example)
print(model_class.eqs)
print("Defaults:", model_class.default_parameters)
print("Initial values:", model_class.default_initial_values)
print(inspect.getsource(model_class.brian2_model))

## Optional reproducible exports

The configuration records model parameters, inputs and integration steps. Set `SAVE_RESULTS=True` to write a summary CSV, configuration JSON and individual raw arrays to `outputs/local-cell-comparison`. No pickle loading is needed.

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    destination = repository / "outputs" / "local-cell-comparison"
    destination.mkdir(parents=True, exist_ok=True)
    summary.to_csv(destination / "summary.csv", index=False)
    (destination / "configuration.json").write_text(
        json.dumps(result["configuration"], indent=2), encoding="utf-8")
    for index, population in enumerate(result["populations"].values()):
        np.savez_compressed(destination / f"model_{index:02d}.npz",
                            time_ms=population["time_ms"],
                            voltage=population["signals"]["v"][:, 0],
                            spike_events_ms=population["spikes_ms"][0])
    print(destination)

## Exercises

1. Run with zero current and identify spontaneous activity or initial-state transients.
2. Change the physical current; use the exported parameters to explain differences in current per capacitance.
3. Halve the PyNN integration step and compare voltage and event timing.
4. Compare each PyNN/native CNT model family. Identify differences in initial conditions, reset rules, parameter scales and event detectors before interpreting them as numerical errors.
5. Use [the original notebook](eg_testing-cell-models-network.ipynb) for synaptic-input and SpiNNaker-specific cell-model comparisons.

## References

- Davison A.P., et al. (2009). PyNN: a common interface for neuronal network simulators. [DOI: 10.3389/neuro.11.011.2008](https://doi.org/10.3389/neuro.11.011.2008).
- Hodgkin A.L., Huxley A.F. (1952). A quantitative description of membrane current and its application to conduction and excitation in nerve. [DOI: 10.1113/jphysiol.1952.sp004764](https://doi.org/10.1113/jphysiol.1952.sp004764).
- Izhikevich E.M. (2003). Simple model of spiking neurons. [DOI: 10.1109/TNN.2003.820440](https://doi.org/10.1109/TNN.2003.820440).
- Brette R., Gerstner W. (2005). Adaptive exponential integrate-and-fire model as an effective description of neuronal activity. [DOI: 10.1152/jn.00686.2005](https://doi.org/10.1152/jn.00686.2005).
- Morris C., Lecar H. (1981). Voltage oscillations in the barnacle giant muscle fiber. [DOI: 10.1016/S0006-3495(81)84782-0](https://doi.org/10.1016/S0006-3495(81)84782-0).
- [Installed-model documentation and parameter units](https://pynn.readthedocs.io/en/latest/reference/neuronmodels.html).

These are model-family references, not a claim that every native default is the original published fit.